# Areal Disaggregation Exercises: Population and Water Stress in Jamaica

In the previous notebook, we walked through a complete areal disaggregation workflow — estimating how many people live in each HUC-8 watershed in the United States by redistributing census tract populations using area-weighted ratios. Now it is your turn.

In this notebook you will apply (and extend) that same workflow using data for **Jamaica**. You have two datasets already loaded in your geodatabase:

| Feature Class | Description | Key Fields |
|---|---|---|
| `Jamaica_hydrobasins` | Watershed boundaries for Jamaica | `HYDROLOGIC` (unique watershed ID), `SURFFACE_RU` (total surface runoff in the watershed) |
| `Jamaica_population` | Parish-level administrative boundaries | `pop_1881`, `pop_1891`, `pop_1911`, `pop_1921`, `pop_1943`, `pop_1960`, `pop_1970`, `pop_1982`, `pop_2001`, `pop_2011`, `pop_2016` |

Notice an important difference from the U.S. exercise: there, the source zones were census tracts (small, many features). Here, the source zones are **parishes** — Jamaica has only 14 parishes, and they are relatively large compared to the watersheds. The uniform-distribution assumption will be weaker for large parishes that contain both dense urban areas and empty mountainous interior. Keep this in mind as you interpret your results.

The three exercises build on each other:

1. **Exercise 1** — Run the basic disaggregation for the 2016 population.
2. **Exercise 2** — Bring in surface runoff data and calculate a water-stress indicator.
3. **Exercise 3** — Generalize your code into a function and loop through all census years.

**How to work through this notebook:** Each exercise has markdown cells that explain what to do, followed by code cells with starter code and comments. Fill in the missing pieces. You are encouraged to refer back to the previous notebook — copying and adapting code is a legitimate and efficient way to work. The skill is knowing **what to change**.

---

## Setup: Imports and Environment

This cell is provided for you. Run it first to import the libraries and set the workspace. Make sure `spaWeek06.gdb` contains both Jamaica datasets.

In [ ]:
# --- Setup (provided) ---

import arcpy
import os
import pandas as pd
import matplotlib.pyplot as plt

arcpy.env.workspace = r"spaWeek06.gdb"
arcpy.env.overwriteOutput = True

print("Workspace: {}".format(arcpy.env.workspace))
print("Setup complete.")

---

# Exercise 1: Basic Disaggregation for Jamaica (2016 Population)

Your goal is to estimate **how many people live in each Jamaican watershed** using the 2016 parish-level population data.

The logic is identical to the U.S. exercise:

1. Record the original area of each source polygon (parish).
2. Intersect the parishes with the watershed boundaries.
3. Calculate the area-weighted estimated population for each sliver.
4. Dissolve by watershed, summing the estimates.

### What you need to think about

- Which feature class is the **source** (has the data to disaggregate) and which is the **target** (defines the zones you want to summarize into)?
- Which **population field** should you use for 2016?
- Which **ID field** uniquely identifies each watershed?
- If you copy the variable definitions from the previous notebook, which variable values need to change? Which can stay the same?

Think about this: the *structure* of the code does not change at all. Only the **variable values** at the top change. This is exactly why we organized all the inputs at the top of the script — it makes the code generalizable.

### Step 1.1 — Define your variables

Fill in the variable values below. Refer to the table at the top of this notebook and think carefully about which field holds the 2016 population.

In [ ]:
# --- Exercise 1, Step 1.1: Define all variables ---

# INPUTS: Fill in the correct feature class names and field names
disaggFC = "___"             # The feature class with population data (source zones)
oPopFieldName = "___"        # The population field for 2016
overlayFC = "___"            # The feature class with watershed boundaries (target zones)
oUnitsfield = "___"          # The unique ID field for each watershed

# PROCESSING
iaFieldName = "Initial_Area"

# TEMPORARY
tintersectFC = r"memory\jamaica_intersect"

# OUTPUTS
EstPopFieldName = "Estimated_Population"
disaggoutFC = "Jamaica_disagg_2016"

# DERIVED
TotalPopFieldName = "{} SUM".format(EstPopFieldName)

# Print your variables to verify
print("Source FC      : {}".format(disaggFC))
print("Pop field      : {}".format(oPopFieldName))
print("Target FC      : {}".format(overlayFC))
print("Target ID field: {}".format(oUnitsfield))

### Step 1.2 — Validate your inputs

Copy or adapt the `validate_inputs` function from the previous notebook. Run it on your two Jamaica datasets to confirm they are both polygons, share a coordinate system, and overlap geographically.

In [ ]:
# --- Exercise 1, Step 1.2: Validate inputs ---

# Copy your validate_inputs function from the previous notebook here,
# then call it with the Jamaica feature classes.




### Step 1.3 — Calculate the original area of each parish

Use `CalculateField` to add and populate the `Initial_Area` field on the source feature class. This records each parish's area before the intersect splits it.

In [ ]:
# --- Exercise 1, Step 1.3: Calculate initial area ---

# Hint: arcpy.management.CalculateField(in_table, field, expression, "PYTHON3", "", field_type)



### Step 1.4 — Run the Pairwise Intersect

Intersect the parish boundaries with the watershed boundaries. Remember to write the result to the in-memory workspace. After the tool finishes, print the feature count so you can see how many slivers were created.

In [ ]:
# --- Exercise 1, Step 1.4: Pairwise Intersect ---




### Step 1.5 — Calculate the estimated population for each sliver

Apply the area-weighted formula. Remember the expression:

$$\text{Estimated Population} = \frac{\text{Sliver Area}}{\text{Original Parish Area}} \times \text{Original Parish Population}$$

Use the variables you defined above — do not hard-code field names into the expression string.

In [ ]:
# --- Exercise 1, Step 1.5: Calculate estimated population ---

# Hint: Build the expression string using .format() and your variable names
# expression = "(!shape.area! / !{}!) * !{}!".format(???, ???)



### Step 1.6 — Dissolve by watershed

Dissolve the slivers by the watershed ID field, summing the estimated populations. This produces your final output — one polygon per watershed with a total estimated population.

In [ ]:
# --- Exercise 1, Step 1.6: Pairwise Dissolve ---




### Step 1.7 — View the results and verify

Print the attribute table of your dissolved output. Then compare the total original population to the total estimated population as a sanity check.

Optionally, add the dissolved layer to the map and symbolize it with graduated colors.

In [ ]:
# --- Exercise 1, Step 1.7: View results and verify totals ---

# Print the attribute table
sum_field = "SUM_{}".format(EstPopFieldName)



# Compare totals



### Step 1.8 — Create a bar chart

Read the dissolved output into a pandas DataFrame and create a horizontal bar chart of estimated population by watershed, just as we did in the previous notebook.

In [ ]:
# --- Exercise 1, Step 1.8: Bar chart of population by watershed ---




---

# Exercise 2: Water Stress Analysis

Now that you have estimated population by watershed, we can go a step further. The `Jamaica_hydrobasins` feature class contains a field called `SURFACE_RU`, which represents the **total surface runoff** for each watershed. Surface runoff is a useful (if simplified) proxy for the **available fresh water supply** in a watershed.

Your task is to combine the population estimate from Exercise 1 with the surface runoff value to calculate a **water-stress indicator** — a number that tells us how much available water there is per person in each watershed.

### The metric

We will calculate:

$$\text{Surface Runoff per Capita} = \frac{\text{SURFFACE\_RU}}{\text{SUM\_Estimated\_Population}}$$

A **lower** value means more people are sharing less water — greater water stress. A **higher** value means more water is available per person. The watershed with the **lowest** surface runoff per capita is under the greatest water stress.

### The problem: the data lives in different feature classes

Your dissolved output from Exercise 1 has the population estimates, but it does **not** have the `SURFFACE_RU` field — that field is in the original `Jamaica_hydrobasins` feature class. You need to bring `SURFFACE_RU` into your dissolved output.

**Hint:** You need to perform a **Join**. Both datasets share the watershed ID field (`HYDROLOGIC`), so you can use `arcpy.management.JoinField` to permanently append `SURFFACE_RU` from the hydrobasins feature class onto your dissolved output. The syntax is:

```python
arcpy.management.JoinField(
    in_data=<your dissolved output>,
    in_field=<the watershed ID field>,
    join_table=<the hydrobasins feature class>,
    join_field=<the same watershed ID field>,
    fields=[<the field(s) you want to bring over>]
)
```

After the join, both `SUM_Estimated_Population` and `SURFFACE_RU` will be in the same table, and you can calculate the per-capita ratio with `CalculateField`.

### Step 2.1 — Join the surface runoff field to your dissolved output

Use `JoinField` to attach `SURFACE_RU` from `Jamaica_hydrobasins` to your Exercise 1 dissolved output. After the join, list the fields to confirm `SURFFACE_RU` is now present.

In [ ]:
# --- Exercise 2, Step 2.1: Join surface runoff to dissolved output ---

# Perform the join


# List fields to confirm
print("Fields after join:")
for field in arcpy.ListFields(disaggoutFC):
    print("  {}".format(field.name))

### Step 2.2 — Calculate Surface Runoff per Capita

Add a new field and calculate the ratio of surface runoff to estimated population. Use the `DOUBLE` field type since this will be a decimal value.

**Be careful:** If any watershed has an estimated population of zero, dividing by it will cause an error. You may want to add a conditional expression or simply verify that no watershed has zero population first.

In [ ]:
# --- Exercise 2, Step 2.2: Calculate runoff per capita ---

runoff_per_cap_field = "Runoff_Per_Capita"

# Hint: the sum field from the dissolve is named "SUM_Estimated_Population"
# expression = "!SURFFACE_RU! / !SUM_Estimated_Population!"



### Step 2.3 — Identify the most water-stressed watershed

Read the results into a pandas DataFrame, sort by `Runoff_Per_Capita`, and print the table. Create a bar chart to visualize the values.

**Answer this question in a comment or a print statement:** Which watershed has the lowest surface runoff per capita, and is therefore under the greatest water stress?

In [ ]:
# --- Exercise 2, Step 2.3: Identify the most water-stressed watershed ---

# Read into a DataFrame


# Sort and print


# Create a bar chart


# Answer: Which watershed is under the greatest water stress?
# print("The most water-stressed watershed is: ___")


---

# Exercise 3: Multi-Year Analysis Using a Loop and a Function

So far, you have run the disaggregation for a single year (2016). But the `Jamaica_population` dataset contains population fields for **eleven different census years**:

`pop_1881`, `pop_1891`, `pop_1911`, `pop_1921`, `pop_1943`, `pop_1960`, `pop_1970`, `pop_1982`, `pop_2001`, `pop_2011`, `pop_2016`

Running the same analysis eleven times by copying and pasting the code would be tedious and error-prone. This is exactly the situation where **functions** and **loops** pay off.

### The plan

1. **Wrap the disaggregation workflow in a function** that takes a population field name as a parameter and returns a DataFrame with the results for that year.
2. **Loop through all eleven years**, calling the function each time.
3. **Collect the results** into a single DataFrame so you can compare populations across years.

### A quick refresher on functions

A function packages a block of reusable code behind a name. You define it with `def`, list its **parameters** (the inputs that change each time), and use `return` to send a result back to the caller.

```python
def disaggregate_population(pop_field, year_label):
    """
    Run the full disaggregation workflow for one population field.
    Returns a DataFrame with columns: HYDROLOGIC, Estimated_Pop, Year
    """
    # ... all the geoprocessing steps go here ...
    # ... read the dissolved result into a DataFrame ...
    # ... add a 'Year' column so we can tell the years apart later ...
    return df
```

Think about what should be **inside** the function (everything that changes per year or that needs to run each time) versus what should stay **outside** (things that are constant, like the workspace and the input feature class names).

### Getting started with the loop

Here is the skeleton of the loop. The list of years and the corresponding field names are provided. Your job is to complete the function body.

```python
years = [1881, 1891, 1911, 1921, 1943, 1960, 1970, 1982, 2001, 2011, 2016]

all_results = []

for year in years:
    pop_field = "pop_{}".format(year)
    print("Processing {}...".format(pop_field))
    df = disaggregate_population(pop_field, year)
    all_results.append(df)

# Combine all years into one DataFrame
combined_df = pd.concat(all_results, ignore_index=True)
```

**Important implementation detail:** Each time the function runs, it will perform an intersect and a dissolve. Make sure you use a **unique output name** for each year's dissolve result (e.g., `"Jamaica_disagg_{}".format(year)`) so that you do not overwrite the previous year's output. The in-memory intersect can be reused each iteration since we set `overwriteOutput = True`.

Also note: the `Initial_Area` field only needs to be calculated **once** on the parish feature class (it does not change between years). You can either calculate it before the loop, or add a check inside the function to skip it if the field already exists.

### Step 3.1 — Calculate Initial Area (once, before the loop)

If you already ran Exercise 1, this field may already exist. It will not hurt to recalculate it — the values will be the same. This ensures the field is ready before the loop starts.

In [ ]:
# --- Exercise 3, Step 3.1: Calculate initial area (once) ---

disaggFC = "Jamaica_population"
overlayFC = "Jamaica_hydrobasins"
oUnitsfield = "HYDROLOGIC"
iaFieldName = "Initial_Area"

arcpy.management.CalculateField(
    in_table=disaggFC,
    field=iaFieldName,
    expression="!shape.area!",
    expression_type="PYTHON3",
    code_block="",
    field_type="DOUBLE"
)

print("Initial_Area calculated on {}".format(disaggFC))

### Step 3.2 — Write the disaggregation function

Complete the function below. It should:

1. Run Pairwise Intersect (parishes × watersheds) — use the in-memory workspace.
2. Calculate the area-weighted estimated population using the provided `pop_field`.
3. Dissolve by `HYDROLOGIC`, summing the estimated population.
4. Read the dissolved result into a pandas DataFrame.
5. Add a `Year` column so we can identify which year each row belongs to.
6. Return the DataFrame.

Most of the code can be adapted directly from what you wrote in Exercise 1. The key difference is that the population field name and the output feature class name are now **parameters** instead of hard-coded values.

In [ ]:
# --- Exercise 3, Step 3.2: Define the disaggregation function ---

def disaggregate_population(pop_field, year):
    """
    Run the area-weighted disaggregation for a single population field
    and return the results as a pandas DataFrame.

    Parameters
    ----------
    pop_field : str  — Name of the population field (e.g., "pop_2016")
    year      : int  — The census year (used for labeling and output naming)

    Returns
    -------
    pandas.DataFrame with columns: HYDROLOGIC, Estimated_Pop, Year
    """
    # Use these variables (already defined above this cell)
    # disaggFC, overlayFC, oUnitsfield, iaFieldName

    EstPopFieldName = "Estimated_Population"
    tintersectFC = r"memory\jamaica_intersect"
    out_fc = "Jamaica_disagg_{}".format(year)
    stat_field = "{} SUM".format(EstPopFieldName)

    # Step A: Pairwise Intersect
    # YOUR CODE HERE


    # Step B: Calculate estimated population for each sliver
    # YOUR CODE HERE


    # Step C: Pairwise Dissolve by watershed
    # YOUR CODE HERE


    # Step D: Read the dissolved output into a DataFrame
    sum_field = "SUM_{}".format(EstPopFieldName)
    rows = []
    with arcpy.da.SearchCursor(out_fc, [oUnitsfield, sum_field]) as cursor:
        for row in cursor:
            rows.append({"HYDROLOGIC": row[0], "Estimated_Pop": row[1]})

    df = pd.DataFrame(rows)
    df["Year"] = year  # Add the year label

    print("  {} complete — {} watersheds".format(pop_field, len(df)))
    return df

### Step 3.3 — Run the loop

The loop below iterates through all eleven census years, calls your function for each one, and collects the results. Once the loop finishes, we concatenate all the individual DataFrames into a single combined DataFrame.

Most of this cell is provided. Make sure your function from Step 3.2 is working correctly before running it.

In [ ]:
# --- Exercise 3, Step 3.3: Loop through all census years ---

years = [1881, 1891, 1911, 1921, 1943, 1960, 1970, 1982, 2001, 2011, 2016]

all_results = []

for year in years:
    pop_field = "pop_{}".format(year)
    print("Processing {}...".format(pop_field))
    df = disaggregate_population(pop_field, year)
    all_results.append(df)

# Combine all years into one DataFrame
combined_df = pd.concat(all_results, ignore_index=True)

print("\nAll years processed.")
print("Combined DataFrame: {} rows x {} columns".format(*combined_df.shape))
print(combined_df.head(20))

### Step 3.4 — Visualize population change over time

Now that you have estimated populations for every watershed across eleven census years, create a **line chart** that shows how population has changed over time. Use matplotlib to plot one line per watershed, with the x-axis as the census year and the y-axis as the estimated population.

**Hint:** You will need to restructure the DataFrame. A **pivot table** is a convenient way to do this:

```python
pivot_df = combined_df.pivot(index="Year", columns="HYDROLOGIC", values="Estimated_Pop")
```

This gives you a table with years as rows and watersheds as columns — perfect for plotting. Then `pivot_df.plot()` will draw one line per column.

If the chart is too crowded with all watersheds, consider plotting only the five most populated watersheds (based on the 2016 values).

In [ ]:
# --- Exercise 3, Step 3.4: Line chart of population over time ---

# Pivot the data


# Create the plot




### Step 3.5 — Reflection

In the cell below, write a brief comment (or print statement) addressing these questions:

1. Which watershed(s) have seen the most population growth over the 135-year period?
2. Are there any watersheds where population has **declined** or remained stable?
3. How does the uniform-distribution assumption affect the reliability of the earlier census years (1881, 1891) versus the more recent ones? Think about how Jamaica's settlement patterns may have changed over that period.
4. What would you change or add to make this analysis more rigorous?

In [ ]:
# --- Exercise 3, Step 3.5: Reflection ---

# Write your answers as comments or print statements below.




---

## You Are Done!

In these exercises you:

- Applied an existing workflow to a new dataset by changing only the input variables.
- Extended the analysis by joining ancillary data and calculating a derived water-stress metric.
- Generalized the workflow into a reusable function and automated it with a loop across eleven census years.

These are fundamental software engineering practices — **parameterization**, **modularity**, and **automation** — and they are the foundation of everything we will build in this course going forward, including custom toolboxes.

In [ ]:
print("Exercises complete!")